# CO3133 — Assignment 3 (M1 Proposal): Multimodal EDA on UPMC Food-101

- **Group:** G-M10 (Trần Gia Lâm - 2352670 [Leader], Nguyễn Hữu Cầu - 2352129 [Member])
- **Instructor:** Lê Thành Sách
- **Task:** Image–Text Multimodal Classification (101 Food Categories)
- **Dataset:** UPMC Food-101 (`kkim0451/UPMC-Food101` on Hugging Face Hub)
- **Leakage-Prevention Unit:** `recipe_id` (paired image + title) grouped by exact SHA-256 and near-duplicate dHash (`dist <= 3`) clusters.

In [ ]:
# 1. Environment Setup & Dataset Download (Colab / Local compatible)
import hashlib
import html
import io
import json
import os
import platform
import re
import sys
import time
import zipfile
from collections import Counter, defaultdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download
from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedGroupKFold

ImageFile.LOAD_TRUNCATED_IMAGES = False

ZIP_PATH = hf_hub_download('kkim0451/UPMC-Food101', 'UPMC-Food-101.zip', repo_type='dataset')
OUT_DIR = '../outputs/a3_eda' if os.path.isdir('../outputs') else 'outputs/a3_eda'
FIG_DIR = os.path.join(OUT_DIR, 'figures')
TAB_DIR = os.path.join(OUT_DIR, 'tables')
SPLIT_DIR = os.path.join(OUT_DIR, 'splits')
for d in [OUT_DIR, FIG_DIR, TAB_DIR, SPLIT_DIR]:
    os.makedirs(d, exist_ok=True)
print('Archive path:', ZIP_PATH)
print('Output dir:', os.path.abspath(OUT_DIR))

In [ ]:
# 2. Load Metadata & Inspect Text Modality Quality
with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    zip_set = set(z.namelist())
    with z.open('UPMC-Food-101/train.csv') as f:
        train_df = pd.read_csv(f)
    with z.open('UPMC-Food-101/test.csv') as f:
        test_df = pd.read_csv(f)

train_df['official_split'] = 'train'
test_df['official_split'] = 'test'
df = pd.concat([train_df, test_df], ignore_index=True)

raw_texts = df['text'].fillna('').astype(str).tolist()
unescaped_texts = [html.unescape(t).strip() for t in raw_texts]
df['clean_text'] = unescaped_texts
df['has_html_entity'] = [r != html.unescape(r) for r in raw_texts]
df['word_count'] = [len(re.findall(r'\w+', t)) for t in unescaped_texts]
df['char_count'] = [len(t) for t in unescaped_texts]
df['is_empty_text'] = [len(t.strip()) == 0 for t in unescaped_texts]
df['class_in_text'] = [
    ann.replace('_', ' ').lower() in txt.lower()
    for ann, txt in zip(df['annotation'], unescaped_texts)
]
print(f'Total raw records: {len(df):,} | Classes: {df["annotation"].nunique()}')

In [ ]:
# 3. Image Integrity, Dimensions, Color Modes & Perceptual Hash (dHash)
def compute_dhash(img: Image.Image) -> int:
    gray = img.convert('L').resize((9, 8), Image.Resampling.BILINEAR)
    arr = np.asarray(gray, dtype=np.int16)
    diff = arr[:, 1:] > arr[:, :-1]
    h = 0
    for b in diff.flatten():
        h = (h << 1) | int(b)
    return h

n = len(df)
img_valid = [False] * n
img_widths, img_heights = [0] * n, [0] * n
img_modes, img_sha256, img_dhash = [''] * n, [''] * n, [None] * n
file_failures = []

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    for i, row in enumerate(df.itertuples(index=False)):
        rel_path = f'UPMC-Food-101/images/{row.official_split}/{row.annotation}/{row.id}'
        if rel_path not in zip_set:
            file_failures.append({'official_split': row.official_split, 'id': row.id, 'annotation': row.annotation, 'error': 'missing_in_zip'})
            continue
        raw_bytes = z.read(rel_path)
        img_sha256[i] = hashlib.sha256(raw_bytes).hexdigest()
        try:
            with Image.open(io.BytesIO(raw_bytes)) as im:
                im.load()
                img_widths[i], img_heights[i] = im.size
                img_modes[i] = im.mode
                img_dhash[i] = compute_dhash(im)
                img_valid[i] = True
        except Exception as e:
            file_failures.append({'official_split': row.official_split, 'id': row.id, 'annotation': row.annotation, 'error': str(e)[:120]})

df['img_valid'] = img_valid
df['width'] = img_widths
df['height'] = img_heights
df['mode'] = img_modes
df['sha256'] = img_sha256
df['dhash'] = img_dhash
df['is_valid_pair'] = df['img_valid'] & (~df['is_empty_text'])
print(f'Valid pairs: {df["is_valid_pair"].sum():,} | Failed images: {len(file_failures)}')

In [ ]:
# 4. Duplicate Detection & Leakage-Free StratifiedGroupKFold Split
def find_near_duplicates(hashes, max_dist=3):
    bands = [defaultdict(list) for _ in range(4)]
    for idx, h in enumerate(hashes):
        if h is None:
            continue
        for b in range(4):
            bands[b][(h >> (16 * b)) & 0xFFFF].append(idx)
    seen, pairs = set(), []
    for b in range(4):
        for bucket in bands[b].values():
            if len(bucket) < 2 or len(bucket) > 200:
                continue
            for i in range(len(bucket)):
                for j in range(i + 1, len(bucket)):
                    u, v = bucket[i], bucket[j]
                    a, c = (u, v) if u < v else (v, u)
                    if (a, c) in seen:
                        continue
                    d = (hashes[a] ^ hashes[c]).bit_count()
                    if d <= max_dist:
                        seen.add((a, c))
                        pairs.append((a, c, d))
    return pairs

near_pairs = find_near_duplicates(img_dhash, max_dist=3)
parent = list(range(len(df)))
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[rb] = ra

sha_groups = defaultdict(list)
for i, h in enumerate(img_sha256):
    if h:
        sha_groups[h].append(i)
for idxs in sha_groups.values():
    for k in range(1, len(idxs)):
        union(idxs[0], idxs[k])
for u, v, _ in near_pairs:
    union(u, v)

df['leakage_group'] = [find(i) for i in range(len(df))]
train_sub = df[(df['official_split'] == 'train') & df['is_valid_pair']].copy()
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(sgkf.split(train_sub, train_sub['label'].values, train_sub['leakage_group'].values))
df['proposed_split'] = 'excluded_invalid'
df.loc[train_sub.index[tr_idx], 'proposed_split'] = 'train'
df.loc[train_sub.index[va_idx], 'proposed_split'] = 'val'
df.loc[(df['official_split'] == 'test') & df['is_valid_pair'], 'proposed_split'] = 'test'
print(df['proposed_split'].value_counts())

## 5. Export Splits, Tables, JSON Summaries, and Figures
All EDA outputs are saved to `outputs/a3_eda/` (`summary.json`, `quality_checks.json`, `tables/*.csv`, `splits/*.csv`, and `figures/*.png`) and consumed directly by `assignment3.md` and `G-M10_A3_Proposal.pdf`.